## Setup

In [ ]:
import torch
import numpy as np
import math
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import sys
import os
from IPython.display import display, HTML

sys.path.insert(0, os.path.abspath('../pytorch-physics/'))
sys.path.insert(0, os.path.abspath('../pytorch-geometric/'))
sys.path.insert(0, os.path.abspath('../utils/'))
from boid_tracking import *
from boid import Flock
from coordinate_orientaitons import *
from helper import html

In [ ]:
# temporary cool, but then one unit
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
N = 2000
box_top = 100

flock_args = {
    'D': 2,
    'N': N,
    'box_top': box_top,
    'pass_through_edges': True,
    'bouncy_edges': False,
    'device': device,
}

boid_args = {
    'init_speed': None,
    # 'min_speed': 3,
    # 'max_speed': 6,
    # 'max_acc': 0.5,

    'min_speed': 3/9,
    'max_speed': 6/9,
    'max_acc': 0.5/9,
    
    'view_radius': 10,
    'view_angle': None,
    
    'avoid_radius': 8,
    'avoid_view': True,
    
    'sep_factor': 0.5,    # avoidfactor
    'align_factor': 0.05,  # matchingfactor
    'cohe_factor': 0.005,  # centeringfactor
    'bias_factor': 0.005,
    'edge_factor': 0.05,
    
    'is_debug': False
}

### HTML

## Visualization

In [ ]:
flock = Flock(
    **flock_args,
    **boid_args
)

In [ ]:
list_of_birds_pos = []
list_of_birds_vel = []

iterations = 100

for _ in range(iterations):
    flock.update()
    list_of_birds_pos.append(flock.pos)
    list_of_birds_vel.append(flock.vel)

vid_len = 50
visualize_boids(list_of_birds_pos[iterations-vid_len:], list_of_birds_vel[iterations-vid_len:], [0, box_top])

In [ ]:
# list_of_birds_pos = []
# list_of_birds_vel = []

# iterations = 200

# for _ in range(iterations):
#     flock.update()
#     list_of_birds_pos.append(flock.pos)
#     list_of_birds_vel.append(flock.vel)

vid_len = 50
visualize_boids(list_of_birds_pos[iterations-vid_len:], list_of_birds_vel[iterations-vid_len:], [40, 60])

In [ ]:
def update_and_calc_flocks(flock, n=5, tracked_indicies=np.array([])):
    flock.update()
    flock_pos = flock.pos.cpu().numpy()
    flock_vel = flock.vel.cpu().numpy()

    if tracked_indicies.shape[0] == 0:
        # choose  one boid from the area [40, 60]^2
        window_flocks = (flock_pos[:, 0] >= 40) & (flock_pos[:, 0] <= 60) & (flock_pos[:, 1] >= 40) & (flock_pos[:, 1] <= 60)
        window_flocks_abs_indicies = np.arange(0, flock_pos.shape[0])[window_flocks]
        tracked_flock_ind = np.random.choice(window_flocks_abs_indicies)
        tracked_flock_pos = flock_pos[tracked_flock_ind]
        # find the n*2-closests boids
        n_nearest_indices = np.argsort(np.linalg.norm(flock_pos - tracked_flock_pos, axis=1))[:(2*n)]
        # choose n unique boids at random
        tracked_indicies = np.random.choice(n_nearest_indices, size=n, replace=False)

    flocks_with_ids = np.column_stack([flock_pos[tracked_indicies], flock_vel[tracked_indicies], tracked_indicies])
    
    return flocks_with_ids

In [ ]:
import matplotlib.colors as mcolors

def visualize_boids_multiple_timeframes(birds_pos, birds_vel, birds_id, boundaries):
    birds_pos = birds_pos.cpu().numpy()
    birds_vel = birds_vel.cpu().numpy()
    birds_id = birds_id.cpu().numpy()
    
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.set_xlim(boundaries[0], boundaries[1])
    ax.set_ylim(boundaries[0], boundaries[1])
    ax.set_xlabel('X-axis')
    ax.set_ylabel('Y-axis')
    
    initial_positions = birds_pos
    initial_velocities = birds_vel
    
    num_birds = len(np.unique(birds_id))
    cmap = plt.cm.get_cmap('rainbow')
    
    norm = mcolors.Normalize(vmin=birds_id.min(), vmax=birds_id.max())
    colors = cmap(norm(birds_id))
    
    scatter = ax.scatter(initial_positions[:, 0], initial_positions[:, 1], c=colors, s=10)
    
    quiver = ax.quiver(initial_positions[:, 0], initial_positions[:, 1], 
                       initial_velocities[:, 0], initial_velocities[:, 1], 
                       color=colors, angles='xy', scale_units='xy', scale=1)
    
    plt.show()
    plt.close(fig)

In [ ]:
n, iterations = 5, 30

tracked_indicies = np.array([])
list_of_birds_pos = []
list_of_birds_vel = []
list_of_birds_ind = []

for i in range(iterations):
    if i % 10 == 0:
        flocks = update_and_calc_flocks(flock, n)
        tracked_indicies = flocks[:, 4].astype(int)
    else:
        flocks = update_and_calc_flocks(flock, n, tracked_indicies)

    list_of_birds_pos.append(torch.tensor(flocks[:, 0:2]))
    list_of_birds_vel.append(torch.tensor(flocks[:, 2:4]))    
    list_of_birds_ind.append(torch.tensor(tracked_indicies))

# vid_len = 200
# visualize_boids(list_of_birds_pos[iterations-vid_len:], list_of_birds_vel[iterations-vid_len:], [30, 70])

In [ ]:
offset = 20
torch_bird_pos = torch.stack(list_of_birds_pos[offset+0:offset+10]).reshape(-1, 2)
torch_bird_vel = torch.stack(list_of_birds_vel[offset+0:offset+10]).reshape(-1, 2)
torch_bird_ind = torch.stack(list_of_birds_ind[offset+0:offset+10]).reshape(-1, 1)

visualize_boids_multiple_timeframes(torch_bird_pos, torch_bird_vel, torch_bird_ind, [30, 70])

## Graph

In [ ]:
# def create_id_tracking_graph_gnn_testing(flocks_pos, flocks_vel, flocks_id):
#     unique_ids = torch.unique(flocks_id)
#     time_factor = torch.
    
#     flock_count_s = flocks_pos_time_s.shape[0]
    
#     indicies_time_s = np.arange(0, flock_count_s)
    
#     nodes_time_s = np.column_stack([indicies_time_s, flocks_pos_time_s, np.zeros(flock_count_s)]).astype(dtype=float)
#     node_features = np.concatenate([nodes_time_s, nodes_time_t])
    
#     edge_connections = np.array(np.meshgrid(indicies_time_t[1:], np.row_stack([indicies_time_s[:], indicies_time_t[:]]))).T.reshape(-1, 2)
#     edge_connections = edge_connections[edge_connections[:, 0] != edge_connections[:, 1]].astype(int)
    
#     dists, angles = dist_angle_from_matrix(np.concatenate([flocks_pos_time_s, flocks_pos_time_t]), edge_connections)
#     vel_rad = velocity_vector_rad(np.row_stack([flocks_vel_time_s, flocks_vel_time_t]), edge_connections)
#     rad_angles = np.deg2rad(angles)
#     edge_features = torch.column_stack([torch.tensor(dists), torch.tensor(rad_angles), torch.tensor(vel_rad)])
#     node_features = torch.tensor(node_features, dtype=torch.float)
#     edge_connections = torch.tensor(edge_connections)
    
#     data = Data(
#         x=node_features,
#         edge_index=edge_connections.t().contiguous(),
#         edge_attr=edge_features
#     )

#     return data

In [ ]:
flocks_pos, flocks_vel, flocks_id = torch_bird_pos, torch_bird_vel, torch_bird_ind
flocks_pos, flocks_vel, flocks_id

In [ ]:
unique_ids = torch.unique(flocks_id)
unique_ids, unique_ids.shape

In [ ]:
torch.arange(0, 5).repeat(3).sort()

In [ ]:
flocks_id.shape[0]